# MAVERIC Data Curation GUI

Interactive Jupyter notebook for visualizing raw retrieval data and setting quality control thresholds.

## Features:
- Load raw retrieval data from `results_dir/datasetName/raw/` folder
- Interactive visualization of quality metric distributions
- Dynamic threshold adjustment with real-time filtering
- Sample image gallery with quality scores
- Export filtered training datasets

## Usage:
1. Configure the dataset and paths in the setup section
2. Run all cells to load data and display the interactive GUI
3. Adjust thresholds and weights using the sliders
4. Visualize sample images to validate filtering
5. Export the final filtered dataset

In [ ]:
# Import required libraries
import os
import sys
import json
import yaml
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO
from pathlib import Path
from typing import Dict, List, Optional, Any

# Jupyter widgets for interactive GUI
import ipywidgets as widgets
from IPython.display import display, clear_output

# Add MAVERIC to path
sys.path.insert(0, str(Path().resolve().parent))

# Import MAVERIC components
from maveric import MAVERIC
from maveric.config import MAVERICConfig
from maveric.core.interfaces import RetrievalResult, QualityResult
from maveric.datasets.elevater_datasets import ELEVATERDataset

print("✅ All libraries imported successfully")

In [ ]:
# Configuration - Update these paths for your setup

# Dataset configuration
TARGET_DATASET = "cifar10"  # Change to your target dataset (cifar10, cifar100, etc.)
CONFIG_FILE = "maveric_config.yaml"  # Path to your MAVERIC config file

# Auto-detect paths from config or use defaults
try:
    with open(CONFIG_FILE, 'r') as f:
        config = yaml.safe_load(f)
    
    # Get paths from config
    RESULTS_DIR = config.get('results_dir', './results')
    RAW_DATA_DIR = f"{RESULTS_DIR}/{TARGET_DATASET}/raw"
    OUTPUT_DIR = f"{RESULTS_DIR}/{TARGET_DATASET}"
    
    print(f"📋 Configuration loaded from: {CONFIG_FILE}")
    print(f"📁 Raw data directory: {RAW_DATA_DIR}")
    print(f"📁 Output directory: {OUTPUT_DIR}")
    
except Exception as e:
    print(f"⚠️  Could not load config file: {e}")
    print("Using default paths...")
    RESULTS_DIR = "./results"
    RAW_DATA_DIR = f"./results/{TARGET_DATASET}/raw"
    OUTPUT_DIR = f"./results/{TARGET_DATASET}"
    config = {}

# Output filename for filtered dataset
OUTPUT_FILENAME = f"{TARGET_DATASET}_training_maveric_gui.json"

In [ ]:
class MAVERICQualityControlGUI:
    """
    Interactive GUI for MAVERIC Quality Control with modern MAVERIC integration
    """
    
    def __init__(self, raw_data_dir: str, target_dataset: str, config: Dict = None):
        """
        Initialize the MAVERIC Quality Control GUI
        
        Args:
            raw_data_dir: Directory containing raw retrieval JSON files
            target_dataset: Target dataset name (e.g., 'cifar10')
            config: MAVERIC configuration dictionary
        """
        self.raw_data_dir = raw_data_dir
        self.target_dataset = target_dataset.lower()
        self.config = config or {}
        self.data = None
        self.filtered_data = None
        
        # Get class names from ELEVATER dataset
        try:
            elevater_dataset = ELEVATERDataset(self.target_dataset)
            self.class_names = elevater_dataset.class_names
            print(f"📋 Using {len(self.class_names)} class names for {self.target_dataset.upper()}")
        except Exception as e:
            print(f"⚠️  Could not load class names: {e}")
            self.class_names = []
        
        # Default metric weights from config or defaults
        self.metric_weights = self.config.get('metric_weights', {
            'img2img': 0.40,
            'txt2txt': 0.20,
            'img2txt': 0.20,
            'txt2img': 0.20
        })
        
        # Default thresholds from config or defaults
        self.thresholds = self.config.get('quality_thresholds', {
            'weighted_class_score': 0.493,
            'consistency': 0.796,
            'resolution_score': 0.370,
            'sharpness_score': 0.880,
            'color_score': 0.768
        })
        
        print(f"🎯 Target dataset: {self.target_dataset}")
        print(f"📁 Raw data directory: {self.raw_data_dir}")
        
        # Load the data
        self.load_data()
    
    def load_data(self):
        """
        Load all raw retrieval JSON files from the directory
        """
        print(f"🔍 Loading raw retrieval data from {self.raw_data_dir}")
        
        # Find all JSON files matching the pattern
        file_pattern = os.path.join(self.raw_data_dir, f"{self.target_dataset}_raw_maveric_*.json")
        json_files = glob.glob(file_pattern)
        
        if not json_files:
            print(f"❌ No files found matching pattern: {file_pattern}")
            print(f"Expected files like: {self.target_dataset}_raw_maveric_1.json")
            return
        
        print(f"📄 Found {len(json_files)} raw data files")
        
        # Load and combine all data
        all_data = []
        for file_path in sorted(json_files):
            try:
                with open(file_path, 'r') as f:
                    file_data = json.load(f)
                    if isinstance(file_data, list):
                        all_data.extend(file_data)
                    else:
                        all_data.append(file_data)
                print(f"  ✅ Loaded {os.path.basename(file_path)}")
            except Exception as e:
                print(f"  ❌ Error loading {file_path}: {e}")
        
        if all_data:
            # Convert to DataFrame for easier processing
            self.data = pd.DataFrame(all_data)
            print(f"📊 Total items loaded: {len(self.data):,}")
            
            # Calculate best class and weighted scores
            self.calculate_best_class()
            
            # Initialize filtered data to all data
            self.filtered_data = self.data.copy()
            
            # Show data summary
            self.show_data_summary()
        else:
            print("❌ No data loaded")
    
    def show_data_summary(self):
        """
        Display summary statistics of the loaded data
        """
        if self.data is None:
            return
        
        print("\n📈 Data Summary:")
        print(f"  • Total samples: {len(self.data):,}")
        
        # Show available metrics
        quality_metrics = [col for col in self.data.columns if any(metric in col for metric in ['score', 'consistency'])]
        print(f"  • Available quality metrics: {len(quality_metrics)}")
        
        # Show class distribution
        if 'label' in self.data.columns:
            class_dist = self.data['label'].value_counts()
            print(f"  • Classes represented: {len(class_dist)}")
            print(f"  • Samples per class (avg): {class_dist.mean():.1f}")
        
        print()
    
    def calculate_best_class(self):
        """
        Calculate the best class for each item using weighted scores
        and add 'label', 'weighted_class_score', and 'consistency' columns
        """
        if self.data is None or not self.class_names:
            print("⚠️  Cannot calculate best class - missing data or class names")
            return
        
        best_classes = []
        weighted_scores = []
        consistency_scores = []
        
        for _, row in self.data.iterrows():
            class_scores = {}
            
            # Calculate weighted score for each class
            for class_name in self.class_names:
                weighted_score = 0.0
                valid_weights_sum = 0.0
                
                # Process each sub-metric
                for metric, weight in self.metric_weights.items():
                    col_name = f"Class_{class_name}_{metric}"
                    
                    if col_name in row and not pd.isna(row[col_name]):
                        weighted_score += row[col_name] * weight
                        valid_weights_sum += weight
                
                # Normalize the score
                if valid_weights_sum > 0:
                    weighted_score /= valid_weights_sum
                
                class_scores[class_name] = weighted_score
            
            # Find the class with the highest weighted score
            if class_scores:
                best_class = max(class_scores.items(), key=lambda x: x[1])
                best_classes.append(best_class[0])
                weighted_scores.append(best_class[1])
                
                # Get consistency score for the best class
                consistency_col = f"Class_{best_class[0]}_consistency"
                consistency_scores.append(row.get(consistency_col, 0.0))
            else:
                # Fallback
                best_classes.append('unknown')
                weighted_scores.append(0.0)
                consistency_scores.append(0.0)
        
        # Add columns to dataframe
        self.data['label'] = best_classes
        self.data['weighted_class_score'] = weighted_scores
        self.data['consistency'] = consistency_scores
        
        print(f"✅ Best class calculated using weighted approach")
    
    def set_metric_weight(self, metric: str, value: float):
        """
        Set a weight for a metric and recalculate best classes
        """
        if metric in self.metric_weights:
            self.metric_weights[metric] = value
            self.calculate_best_class()
    
    def set_threshold(self, metric: str, value: float):
        """
        Set a threshold for a quality metric
        """
        if metric in self.thresholds:
            self.thresholds[metric] = value
    
    def apply_thresholds(self) -> int:
        """
        Apply the current thresholds to filter the data
        
        Returns:
            Number of samples after filtering
        """
        if self.data is None:
            return 0
        
        # Start with all data
        self.filtered_data = self.data.copy()
        
        # Apply visual quality thresholds
        for metric in ['resolution_score', 'sharpness_score', 'color_score']:
            if metric in self.thresholds and metric in self.filtered_data.columns:
                self.filtered_data = self.filtered_data[
                    self.filtered_data[metric] >= self.thresholds[metric]
                ]
        
        # Apply class-specific thresholds
        mask = pd.Series(True, index=self.filtered_data.index)
        
        for _, row in self.filtered_data.iterrows():
            # Check weighted class score
            if ('weighted_class_score' in row and 
                row['weighted_class_score'] < self.thresholds['weighted_class_score']):
                mask.loc[row.name] = False
                continue
            
            # Check consistency score
            if ('consistency' in row and 
                row['consistency'] < self.thresholds['consistency']):
                mask.loc[row.name] = False
                continue
        
        # Apply the mask
        self.filtered_data = self.filtered_data[mask]
        
        return len(self.filtered_data)
    
    def visualize_metrics_distribution(self, metrics: List[str] = None):
        """
        Visualize the distribution of metrics with thresholds
        """
        if self.data is None:
            print("❌ No data loaded")
            return
        
        if metrics is None:
            metrics = ['weighted_class_score', 'consistency', 'resolution_score', 
                      'sharpness_score', 'color_score']
        
        # Filter metrics that exist in the data
        available_metrics = [m for m in metrics if m in self.data.columns]
        
        if not available_metrics:
            print("❌ No valid metrics to visualize")
            return
        
        # Create subplots
        n_metrics = len(available_metrics)
        fig, axes = plt.subplots(n_metrics, 1, figsize=(12, 4*n_metrics))
        
        if n_metrics == 1:
            axes = [axes]
        
        for i, metric in enumerate(available_metrics):
            ax = axes[i]
            data = self.data[metric].dropna()
            
            if len(data) > 0:
                # Calculate statistics
                mean_val = data.mean()
                std_val = data.std()
                threshold = self.thresholds.get(metric, 0)
                
                # Plot histogram
                ax.hist(data, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
                
                # Plot threshold line
                ax.axvline(threshold, color='red', linestyle='--', linewidth=2,
                          label=f'Threshold: {threshold:.3f}')
                
                # Plot mean line
                ax.axvline(mean_val, color='green', linestyle='-', linewidth=2,
                          label=f'Mean: {mean_val:.3f}')
                
                # Plot std lines
                ax.axvline(mean_val - std_val, color='blue', linestyle=':', linewidth=1,
                          label=f'Mean-Std: {(mean_val-std_val):.3f}')
                ax.axvline(mean_val + std_val, color='blue', linestyle=':', linewidth=1,
                          label=f'Mean+Std: {(mean_val+std_val):.3f}')
                
                ax.set_title(f'Distribution of {metric}', fontsize=14, fontweight='bold')
                ax.set_xlabel('Value', fontsize=12)
                ax.set_ylabel('Count', fontsize=12)
                ax.legend()
                ax.grid(True, alpha=0.3)
            else:
                ax.text(0.5, 0.5, f"No data for {metric}",
                       ha='center', va='center', transform=ax.transAxes)
        
        plt.tight_layout()
        plt.show()
    
    def visualize_sample_images(self, n_samples: int = 5, random_seed: int = 42):
        """
        Visualize sample images from the filtered data
        """
        if self.filtered_data is None or len(self.filtered_data) == 0:
            print("❌ No filtered data available")
            return
        
        # Set random seed for reproducibility
        np.random.seed(random_seed)
        
        # Select random samples
        if len(self.filtered_data) <= n_samples:
            samples = self.filtered_data
        else:
            sample_indices = np.random.choice(
                len(self.filtered_data), n_samples, replace=False)
            samples = self.filtered_data.iloc[sample_indices]
        
        # Create figure
        fig, axes = plt.subplots(1, len(samples), figsize=(4*len(samples), 5))
        
        if len(samples) == 1:
            axes = [axes]
        
        # Load and display images
        for i, (_, row) in enumerate(samples.iterrows()):
            try:
                # Get image URL
                url = row['url']
                
                # Download image with timeout
                response = requests.get(url, timeout=10)
                image = Image.open(BytesIO(response.content)).convert('RGB')
                
                # Display image
                axes[i].imshow(image)
                
                # Create title with metrics
                label = row['label']
                metrics_text = f"ID: {row.get('id', i)}\n"
                metrics_text += f"Label: {label}\n"
                metrics_text += f"Weighted: {row.get('weighted_class_score', 0):.3f}\n"
                metrics_text += f"Consistency: {row.get('consistency', 0):.3f}\n"
                metrics_text += f"Resolution: {row.get('resolution_score', 0):.3f}\n"
                metrics_text += f"Sharpness: {row.get('sharpness_score', 0):.3f}"
                
                axes[i].set_title(metrics_text, fontsize=9)
                axes[i].axis('off')
                
            except Exception as e:
                axes[i].text(0.5, 0.5, f"Error loading image:\n{str(e)[:50]}...",
                           ha='center', va='center', transform=axes[i].transAxes,
                           fontsize=10, bbox=dict(boxstyle="round", facecolor='wheat'))
                axes[i].axis('off')
        
        plt.tight_layout()
        plt.show()
    
    def save_filtered_data(self, output_dir: str, filename: str = None) -> str:
        """
        Save the filtered data in MAVERIC training dataset format
        """
        if self.filtered_data is None or len(self.filtered_data) == 0:
            print("❌ No filtered data to save")
            return None
        
        if filename is None:
            filename = f"{self.target_dataset}_training_maveric_gui.json"
        
        output_path = os.path.join(output_dir, filename)
        
        try:
            # Create output directory if needed
            os.makedirs(output_dir, exist_ok=True)
            
            # Format data for training
            training_data = []
            
            for _, row in self.filtered_data.iterrows():
                item = {
                    'id': int(row.get('id', 0)),
                    'url': str(row.get('url', '')),
                    'label': str(row.get('label', '')),
                    'text': str(row.get('text', '')),
                    'weighted_class_score': float(row.get('weighted_class_score', 0.0)),
                    'consistency': float(row.get('consistency', 0.0))
                }
                training_data.append(item)
            
            # Save to JSON
            with open(output_path, 'w') as f:
                json.dump(training_data, f, indent=2)
            
            print(f"💾 Saved {len(training_data):,} filtered items to: {output_path}")
            return output_path
            
        except Exception as e:
            print(f"❌ Error saving filtered data: {e}")
            return None

# Initialize the quality control system
print("🚀 Initializing MAVERIC Quality Control GUI...")
qc = MAVERICQualityControlGUI(
    raw_data_dir=RAW_DATA_DIR,
    target_dataset=TARGET_DATASET,
    config=config
)

In [ ]:
def create_interactive_gui():
    """
    Create the interactive GUI using ipywidgets
    """
    # Create threshold widgets
    threshold_widgets = {}
    
    for metric, default_value in qc.thresholds.items():
        # Adjust max value based on metric type
        max_value = 1.0
        if 'resolution' in metric:
            max_value = 5.0
        
        # Clean up metric name for display
        display_name = metric.replace('_', ' ').title()
        
        threshold_widgets[metric] = widgets.FloatSlider(
            value=default_value,
            min=0.0,
            max=max_value,
            step=0.001,
            description=f'{display_name}:',
            disabled=False,
            continuous_update=False,
            orientation='horizontal',
            readout=True,
            readout_format='.3f',
            layout=widgets.Layout(width='500px'),
            style={'description_width': '200px'}
        )
    
    # Create metric weight widgets
    weight_widgets = {}
    
    for metric, default_value in qc.metric_weights.items():
        display_name = metric.replace('2', ' → ').upper()
        
        weight_widgets[metric] = widgets.FloatSlider(
            value=default_value,
            min=0.0,
            max=1.0,
            step=0.01,
            description=f'{display_name}:',
            disabled=False,
            continuous_update=False,
            orientation='horizontal',
            readout=True,
            readout_format='.2f',
            layout=widgets.Layout(width='500px'),
            style={'description_width': '200px'}
        )
    
    # Create tabs for different settings
    tab = widgets.Tab()
    tab.children = [
        widgets.VBox(list(threshold_widgets.values())),
        widgets.VBox(list(weight_widgets.values()))
    ]
    tab.set_title(0, 'Quality Thresholds')
    tab.set_title(1, 'Metric Weights')
    
    # Create control buttons
    apply_button = widgets.Button(
        description='Apply & Visualize',
        button_style='primary',
        icon='refresh',
        layout=widgets.Layout(width='150px')
    )
    
    samples_button = widgets.Button(
        description='Show Samples',
        button_style='info',
        icon='image',
        layout=widgets.Layout(width='150px')
    )
    
    save_button = widgets.Button(
        description='Save Dataset',
        button_style='success',
        icon='save',
        layout=widgets.Layout(width='150px')
    )
    
    # Status display
    status_display = widgets.HTML(
        value=f"<p><b>Status:</b> Loaded {len(qc.data):,} samples</p>"
    )
    
    # Output area
    output_area = widgets.Output()
    
    # Button callback functions
    def on_apply_clicked(b):
        with output_area:
            clear_output(wait=True)
            
            print("🔄 Updating settings...")
            
            # Update metric weights
            for metric, widget in weight_widgets.items():
                qc.set_metric_weight(metric, widget.value)
            
            # Update thresholds
            for metric, widget in threshold_widgets.items():
                qc.set_threshold(metric, widget.value)
            
            # Apply filtering
            count = qc.apply_thresholds()
            retention_rate = (count / len(qc.data)) * 100 if len(qc.data) > 0 else 0
            
            status_display.value = (
                f"<p><b>Filtered:</b> {count:,} samples "
                f"({retention_rate:.1f}% retention rate)</p>"
            )
            
            print(f"✅ Applied filters: {count:,} samples retained ({retention_rate:.1f}%)")
            
            # Show updated distribution
            qc.visualize_metrics_distribution()
    
    def on_samples_clicked(b):
        with output_area:
            clear_output(wait=True)
            print("🖼️  Loading sample images...")
            qc.visualize_sample_images(n_samples=5)
    
    def on_save_clicked(b):
        with output_area:
            clear_output(wait=True)
            print("💾 Saving filtered dataset...")
            
            save_path = qc.save_filtered_data(
                output_dir=OUTPUT_DIR,
                filename=OUTPUT_FILENAME
            )
            
            if save_path:
                print(f"✅ Dataset saved successfully!")
                print(f"📁 Location: {save_path}")
                print(f"📊 Items: {len(qc.filtered_data):,}")
    
    # Connect button callbacks
    apply_button.on_click(on_apply_clicked)
    samples_button.on_click(on_samples_clicked)
    save_button.on_click(on_save_clicked)
    
    # Create layout
    button_box = widgets.HBox([
        apply_button, samples_button, save_button
    ], layout=widgets.Layout(justify_content='center'))
    
    # Main GUI layout
    gui_layout = widgets.VBox([
        widgets.HTML(value="<h2>🎯 MAVERIC Quality Control Dashboard</h2>"),
        widgets.HTML(value=f"<p><b>Dataset:</b> {TARGET_DATASET.upper()} | "
                           f"<b>Raw Data:</b> {RAW_DATA_DIR}</p>"),
        widgets.HTML(value="<p>Adjust thresholds and weights to filter the dataset. "
                           "Use <b>Apply & Visualize</b> to see the effects.</p>"),
        tab,
        status_display,
        button_box,
        output_area
    ])
    
    # Display the GUI
    display(gui_layout)
    
    # Show initial visualization
    with output_area:
        print("📊 Initial data distribution:")
        qc.visualize_metrics_distribution()

# Only create GUI if data was loaded successfully
if qc.data is not None:
    create_interactive_gui()
else:
    print("❌ Cannot create GUI - no data loaded")
    print("\n🔧 Troubleshooting:")
    print(f"1. Check that raw data files exist in: {RAW_DATA_DIR}")
    print(f"2. Expected file pattern: {TARGET_DATASET}_raw_maveric_*.json")
    print(f"3. Verify the TARGET_DATASET name is correct: {TARGET_DATASET}")
    print(f"4. Check the CONFIG_FILE path: {CONFIG_FILE}")

## Usage Instructions

### 1. Quality Thresholds Tab
- **Weighted Class Score**: Minimum score for the best-matching class
- **Consistency**: Minimum consistency score between image and text
- **Resolution Score**: Minimum image resolution quality
- **Sharpness Score**: Minimum image sharpness
- **Color Score**: Minimum color diversity

### 2. Metric Weights Tab
- **IMG → IMG**: Weight for image-to-image similarity
- **TXT → TXT**: Weight for text-to-text similarity  
- **IMG → TXT**: Weight for image-to-text consistency
- **TXT → IMG**: Weight for text-to-image consistency

### 3. Controls
- **Apply & Visualize**: Update filtering and show distribution plots
- **Show Samples**: Display sample images with quality scores
- **Save Dataset**: Export filtered data as training dataset JSON

### 4. Tips
- Start with default thresholds and adjust based on the distributions
- Use the sample images to validate that filtering is working correctly
- Higher thresholds = stricter filtering = fewer but higher quality samples
- Balance retention rate vs quality based on your training needs